# Guided placement pilot (v6)

Reuse all three trained v5 policies. Compare direct inference, top-4/top-8
simulation, random subsets, full-pool search and RL/heuristic-started local search.
No retraining, Ray or real distributed workload. See `docs/PILOT_V6.md`.

**Unlimited runtime by default**, with resumable progress and parallel CPU cases.
Start with small validation cases; then change STAGE to scale. Keep test separate.


In [ ]:
from pathlib import Path
import os, sys, json
# Limit numerical library threads BEFORE importing NumPy; each simulator process uses one CPU.
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "1"
ROOT = Path.cwd()
if not (ROOT / "learnedplacement").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "learnedplacement").is_dir()
sys.path.insert(0, str(ROOT))
# Install once: python -m pip install -e '.[notebook,learning]'
import pandas as pd
from learnedplacement.runner import read_json
from learnedplacement.guided_runner import (run, status, export, plot_results,
    find_model_run, method_specs, validate_config)


## Source policies

Use your completed **v5 results folder**, not the analysis ZIP. Auto-discovery
requires exactly one distinct completed model set. If it reports ambiguity, set
SOURCE_RUN explicitly. Original files remain unchanged. Restart the kernel after
pulling code. No new training is performed.


In [ ]:
# Set explicitly if needed, e.g. ROOT / "results" / "pilot_v5_8192"
SOURCE_RUN = None
if SOURCE_RUN is None:
    SOURCE_RUN = find_model_run(ROOT / "results")
print("Frozen models from:", SOURCE_RUN)
source_manifest = read_json(SOURCE_RUN / "manifest.json")
print("Training episodes:", source_manifest["config"]["learning"]["episodes"])
print("Policy seeds:", source_manifest["config"]["learning"]["seeds"])


## Editable experiment settings

Small: 6/12/24 available machines (4/8/16 reference active).
Scale: 32/64/96 available machines (24/48/64 reference active).
Each stage/split has 48 cases, 21 methods and 256 paired evaluation traces.
Identical selected plans share evaluation work. Four workers is a conservative
starting point; choose a worker count before starting and keep it fixed on resume.
Use a new output with WORKERS=1 for isolated planning latency measurements.

MAX_MINUTES=None removes the experiment cutoff. The 5-second *method budget*
is retained for the equal-budget scientific comparison; edit its list if desired.


In [ ]:
STAGE = "small"       # "small" first, then "scale"
SPLIT = "validation"  # Reserve "test" until experiment choices are frozen
WORKERS = min(4, os.cpu_count() or 1)
MAX_MINUTES = None    # Unlimited; optionally set any positive number
RUN_NAME = f"pilot_v6_{STAGE}_{SPLIT}_w{WORKERS}"

CONFIG = read_json(ROOT / "configs" / "pilot_v6.json")
CONFIG["split"] = SPLIT
CONFIG["planning_traces"] = 32
CONFIG["evaluation_traces"] = 256
CONFIG["guided"]["top_k"] = [4, 8]
CONFIG["guided"]["candidate_budgets"] = [32]  # e.g. [8, 32, 64]
CONFIG["guided"]["seconds_budgets"] = [5]     # e.g. [1, 5, 20]
CONFIG["learning"]["seeds"] = source_manifest["config"]["learning"]["seeds"]
CONFIG["learning"]["penalties"] = source_manifest["config"]["learning"]["penalties"]
if STAGE == "scale":
    CONFIG["active_nodes"] = [24, 48, 64]
    CONFIG["available_nodes"] = {"24": 32, "48": 64, "64": 96}
    CONFIG["choices"]["replicas"] = [1, 2, 4, 6, 8, 12, 16]
elif STAGE != "small":
    raise ValueError("Choose small or scale")
CONFIG["methods"] = list(method_specs(CONFIG))
validate_config(CONFIG)
OUTPUT = ROOT / "results" / RUN_NAME
print("Output:", OUTPUT)
print("Workers:", WORKERS, "Optional time limit:", MAX_MINUTES)
print(json.dumps(CONFIG, indent=2))


## Run / resume

This is the only cell that runs the experiment. Rerun unchanged to resume.
Independent cases run in CPU processes; each saves its own progress. An interrupt
may wait for active process jobs. Do not rerun while previous workers remain alive.


In [ ]:
result = run(config=CONFIG, output=OUTPUT, source_run=SOURCE_RUN,
             workers=WORKERS, max_minutes=MAX_MINUTES)
print(json.dumps(result, indent=2))


## Saved progress and results (read only)


In [ ]:
print(json.dumps(status(OUTPUT), indent=2))
if (OUTPUT / "summary.csv").exists():
    results = pd.read_csv(OUTPUT / "summary.csv")
    display(results[["case_id", "available_nodes", "method", "restricted_mean_s",
                     "mean_observed_cost_usd", "statistical_screen_passed",
                     "censored_traces", "planning_s", "simulations", "unscored_fallback"]])


## Comparison plots (read only)

Only fully matched cases enter plots. Set AVAILABLE_NODES=None for size curves.
Execution times include failures. Search timings include initialization; cold
model loading/shared trace generation are excluded. Parallel contention affects
wall budgets, so report WORKERS with timing results. Each output has one split.


In [ ]:
PROFILE = "reference"  # Or slow_io, slow_network, variable_repair, None
AVAILABLE_NODES = 12 if STAGE == "small" else 64
if (OUTPUT / "summary.json").exists():
    display(plot_results(OUTPUT, profile=PROFILE, available_nodes=AVAILABLE_NODES))


## Compact analysis export

Share this ZIP after completing or pausing. Models, raw observations and candidate
audits stay local. The export includes selected plans, paired statistics,
completed/attempted planning work, hashes and timing metadata.


In [ ]:
archive = export(OUTPUT)
print("Share:", archive)
print("Size (MiB):", round(archive.stat().st_size / 1024**2, 3))
